# Importing

In [ ]:
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                           Importing                             #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
! source /home/ivm/envs/valid_env/bin/activate
import polars as pl
import sys
sys.path.append(("/home/ivm/valid/scripts/utils/"))
from general_utils import *
from model_eval_utils import compare_models

# For shap
from plot_utils import get_plot_names
import pickle
import shap

%load_ext autoreload
%autoreload 2

# Settings

In [ ]:
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                           Settings                              #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
base_path="/home/ivm/valid/data/processed_data/pred_2026-01"
%env base_path=$base_path
%env fg_ver=R13
%env lab_name=egfr
%env extra=d1_KDIGO-soft2_ld
%env diff=90
%env lab_name_three=uacr
%env extra_three=d1_ld
%env date_three=2025-11-05
%env lab_name_two=cystc
%env extra_two=d1_KDIGO-strict_ld
%env date_two=2025-11-05
%env extra_labels=testv10_2022_w3
%env date_1=2025-11-05
%env date_2=2026-01-07
%env date_3=2026-01-07
%env date_4=2026-01-07
%env date_5=2026-01-07
base_date = datetime(2021,10,1)
%env base_date=2021-10-01
%env start_pred_date=2022-01-01
%env end_pred_date=2022-12-31
%env min_age=30
%env max_age=70
%env months_buffer=3
%env abnorm_type=strong
%env valid_pct=0.3
%env test_pct=0.0
%env version=v10
%env bin_goal=y_MEAN_ABNORM
%env cont_goal=y_MEAN

goal="y_MEAN_ABNORM"
file_descr = "testv10_2022_w3"
lab_name = "egfr"
base_date = datetime(2021,10,1)


# Filters

In [ ]:
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                 Filters                                                 #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
data = pl.read_parquet(base_path+"/step1_clean/egfr_d1_KDIGO-soft2_ld_2025-11-05.parquet")

train_goals = {"mglogloss": "y_MEAN_ABNORM", "logloss": "y_MEAN_ABNORM",  "q90": "y_MEAN", "q95":"y_MEAN", "q75": "y_MEAN", "q25": "y_MEAN", "q10": "y_MEAN", "mae": "y_MEAN"}

lastval_long_filter = (pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date).filter((pl.col.DATE==pl.col.DATE.max()).over("FINNGENID")).filter(pl.col.DATE.dt.year()<2020)["FINNGENID"]))
no_history_filter = (~pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date)["FINNGENID"]))
history_filter = (pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date)["FINNGENID"]))
last_norm_filter = (pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date).filter((pl.col.DATE==pl.col.DATE.max()).over("FINNGENID")).filter(pl.col.ABNORM_CUSTOM<1)["FINNGENID"]))
no_abnorm_filter = (~pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date).filter(pl.col.ABNORM_CUSTOM==1)["FINNGENID"]))
thirty_filter = (((pl.col.EVENT_AGE>=30)&(pl.col.EVENT_AGE<40)))
fourty_filter = (((pl.col.EVENT_AGE>=40)&(pl.col.EVENT_AGE<50)))
fifty_filter = (((pl.col.EVENT_AGE>=50)&(pl.col.EVENT_AGE<60)))
sixty_filter = (((pl.col.EVENT_AGE>=60)&(pl.col.EVENT_AGE<=70)))

set_names = {0: "Train", 1: "Valid", 2: "Test"}
test_filter = pl.col.SET==2
valid_filer = pl.col.SET==1
train_filter = pl.col.SET==0

filters = {"All": True, 
           "History": history_filter, 
           "All normal": no_abnorm_filter|no_history_filter, 
           "Last <2020": lastval_long_filter|no_history_filter, 
           "No history": no_history_filter, 
           "30-40": thirty_filter, 
           "40-50": fourty_filter, 
           "50-60": fifty_filter, 
           "60-70": sixty_filter}

preds_descrs={"1_lab_manualquants_month": "lab sequence", "1_clinpheno": "clinical phenotype", "2_lastval": "last value", "2_sumstats": "sumstats", "3_twosumstats": "eGFR+Cystatin C", "3_twosumstats_2": "eGFR+UACR", "3_otherlabs": "other labs", "3_registry": "registry data", "4_all": "all data", "5_icd": "ICD data", "5_atc": "ATC data"}
metrics = ["logloss", "q10", "q25", "mae"]

goal_names = {"y_MEAN_ABNORM": "Mean abnormal", "y_NEXT_ABNORM": "Next abnormal"}
goal_names_extra = {"y_MEAN_ABNORM": "Mean abnormal", "y_NEXT_ABNORM": "Next abnormal", "y_MEAN": "Mean"}


# Step 0&1 - Extraction and cleaning

## Kreatinine

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step0_extract.py \
    --omop=3020564 \
    --res_dir=/home/ivm/valid/data/processed_data/poster/step0_extract/ \
    --lab_name=krea


In [ ]:
###### Check abnormality medians
extract_data = pl.read_parquet("/home/ivm/valid/data/processed_data/poster/step0_extract/krea_2025-11-05.parquet")
(extract_data
 .with_columns(pl.when(pl.col.ABNORM=="AA").then(pl.lit("HH")).otherwise(pl.col.ABNORM).alias("ABNORM"))
 .with_columns(pl.when(pl.col.ABNORM=="A").then(pl.lit("H")).otherwise(pl.col.ABNORM).alias("ABNORM"))
 .filter(~pl.col.VALUE.is_null(), pl.col.UNIT=="umol/l").group_by("ABNORM").agg(pl.col.VALUE.mean().alias("MEAN"), pl.col.VALUE.median().alias("MEDIAN"))
)


In [ ]:

###### Cleaning
! python3 /home/ivm/valid/scripts/steps/step1_clean.py \
    --res_dir=/home/ivm/valid/data/processed_data/poster/step1_clean/ \
    --file_path=/home/ivm/valid/data/processed_data/poster/step0_extract/krea_2025-11-05.parquet \
    --lab_name=egfr \
    --fill_missing 1 \
    --dummies 48 71 115 -1 625 \
    --abnorm_type=KDIGO-soft2 \
    --main_unit umol/l \
    --plot 1 \
    --keep_last_of_day 1 \
    --ref_min 2 \
    --max_z 4

## Albrumine/Creatinine in Urine

In [ ]:
# Albumine/Creatinine in Urine
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step0_extract.py \
    --omop=3020682 \
    --res_dir=/home/ivm/valid/data/processed_data/poster/step0_extract/ \
    --lab_name=uacr


In [ ]:
###### Check abnormality medians
extract_data = pl.read_parquet("/home/ivm/valid/data/processed_data/poster/step0_extract/uacr_2025-11-05.parquet")
(extract_data
 .with_columns(pl.when(pl.col.ABNORM=="AA").then(pl.lit("HH")).otherwise(pl.col.ABNORM).alias("ABNORM"))
 .with_columns(pl.when(pl.col.ABNORM=="A").then(pl.lit("H")).otherwise(pl.col.ABNORM).alias("ABNORM"))
 .filter(~pl.col.VALUE.is_null(), pl.col.UNIT=="mg/mmol").group_by("ABNORM").agg(pl.col.VALUE.mean().alias("MEAN"), pl.col.VALUE.median().alias("MEDIAN"), pl.len().alias("N")))


In [ ]:

###### Cleaning
! python3 /home/ivm/valid/scripts/steps/step1_clean.py \
    --res_dir=/home/ivm/valid/data/processed_data/poster/step1_clean/ \
    --file_path=/home/ivm/valid/data/processed_data/poster/step0_extract/uacr_2025-11-05.parquet \
    --lab_name=uacr \
    --fill_missing 1 \
    --dummies -1 0.6 16.1 -1 -1 \
    --main_unit mg/mmol \
    --plot 1 \
    --keep_last_of_day 1 \
    --ref_min 0.01 


## Cystatin C

In [ ]:
# Cystatin C
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step0_extract.py \
    --omop=3030366 \
    --res_dir=/home/ivm/valid/data/processed_data/poster/step0_extract/ \
    --lab_name=cystc


In [ ]:

###### Check abnormality medians
extract_data = pl.read_parquet("/home/ivm/valid/data/processed_data/poster/step0_extract/cystc_2025-11-05.parquet")
(extract_data
 .with_columns(pl.when(pl.col.ABNORM=="AA").then(pl.lit("HH")).otherwise(pl.col.ABNORM).alias("ABNORM"))
 .with_columns(pl.when(pl.col.ABNORM=="A").then(pl.lit("H")).otherwise(pl.col.ABNORM).alias("ABNORM"))
 .filter(~pl.col.VALUE.is_null(), pl.col.UNIT=="mg/l").group_by("ABNORM").agg(pl.col.VALUE.mean().alias("MEAN"), pl.col.VALUE.median().alias("MEDIAN"))
)


In [ ]:

###### Cleaning
! python3 /home/ivm/valid/scripts/steps/step1_clean.py \
    --res_dir=/home/ivm/valid/data/processed_data/poster/step1_clean/ \
    --file_path=/home/ivm/valid/data/processed_data/poster/step0_extract/cystc_2025-11-05.parquet \
    --lab_name=cystc \
    --fill_missing 1 \
    --dummies 0.61 0.89 1.79 -1 -1 \
    --abnorm_type=KDIGO-strict \
    --main_unit mg/l \
    --plot 1 \
    --keep_last_of_day 1 \
    --ref_min 2 \
    --transform_egfr 1

# Step 2 - Other diagnosis data

In [ ]:

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                      Step 2 - Other diagnosis data              #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
! python3 /home/ivm/valid/scripts/steps/step2_diags.py \
                --lab_name=egfr \
                --res_dir=/home/ivm/valid/data/processed_data/poster/step2_diags/  \
                --diag_regex="(^N18)|(^N19)|(^Z905)|(^Q600)" --med_regex="(^A10BK)"\
                --diag_excl_regex="" \
                --med_excl_regex="" \
                --fg_ver="R13"


# Step 3 - Exclusions

In [ ]:

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                      Step 3 - Exclusions                        #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
# diag_regex = "(^KAD00)|(^KAD10)|(^KAD01)|(^KAC11)|(^KAC00)|(^KAC01)"
# table = f"`finngen-production-library.sandbox_tools_r13.finngen_r13_service_sector_detailed_longitudinal_v1`"
# query = f"""SELECT FINNGENID, SOURCE, EVENT_AGE, APPROX_EVENT_DAY, CODE1, CODE2
#                              FROM {table}
#                              WHERE (SOURCE IN ("OPER_IN", "OPER_OUT") AND REGEXP_CONTAINS(CODE1, "{diag_regex}")) OR
#                                    (SOURCE IN ("INPAT", "OUTPAT", "PRIM_OUT") AND REGEXP_CONTAINS(CODE2, "^Z905")) OR
#                                    (SOURCE IN ("INPAT", "OUTPAT", "PRIM_OUT") AND REGEXP_CONTAINS(CODE1, "^Z905"))

#                                    """
# nephro = query_to_df(query)

# # CODE1 is diagnosis and CODE2 symptom
# nephro =nephro.unpivot(index=["FINNGENID", "EVENT_AGE", "APPROX_EVENT_DAY", "SOURCE"])
# nephro = nephro.drop(["variable"]).unique()
# nephro = nephro.rename({"value":"CODE"})
# # Other codes are numbers
# nephro = nephro.filter(pl.col("CODE").is_not_null())
# nephro.write_parquet("/home/ivm/valid/data/extra_data/processed_data/step0_extract/nephro_opercodes.parquet")
# #diags = diags.filter(pl.col("CODE").str.contains("^[A-Z][0-9]"))
# print_count(nephro)
# display(nephro)

nephro = pl.read_parquet("/home/ivm/valid/data/extra_data/processed_data/step0_extract/nephro_opercodes.parquet")


In [ ]:
########### ############# ############## ############## ###########
diags_data = pl.read_parquet("/home/ivm/valid/data/processed_data/preds_2026-01/step2_diags/egfr_R13_2025-11-05_diags.parquet")
meds_data = pl.read_parquet("/home/ivm/valid/data/processed_data/preds_2026-01/step2_diags/egfr_R13_2025-11-05_meds.parquet")

egfr_data = pl.read_parquet("/home/ivm/valid/data/processed_data/preds_2026-01/step1_clean/egfr_d1_KDIGO-soft2_ld_2025-11-05.parquet")
cystc_data = pl.read_parquet("/home/ivm/valid/data/processed_data/preds_2026-01/step1_clean/cystc_d1_KDIGO-strict_ld_2025-11-05.parquet")
uacr_data = pl.read_parquet("/home/ivm/valid/data/processed_data/preds_2026-01/step1_clean/uacr_d1_ld_2025-11-05.parquet")


In [ ]:
########### ############# ############## ############## ###########
from diag_utils import get_abnorm_start_dates, get_data_diags

egfr_diag = get_data_diags(get_abnorm_start_dates(egfr_data.filter(pl.col.ABNORM_CUSTOM!=0.5)), 90)


## Numbers

In [ ]:

# Abnormal eGFR
########### ############# ############## ############## ###########

print("Total people")
print_count(egfr_data)
print("Total historical data")
print_count(egfr_data.filter(pl.col.DATE<base_date))
print("History of abnormal eGFR")
print_count(egfr_data
 .filter(pl.col.DATE<base_date)
 .filter(((pl.col.ABNORM_CUSTOM!=0).any().over("FINNGENID")))
)
print("History of two abnormal eGFR")
print_count(egfr_diag
 .filter(pl.col.DATA_DIAG_DATE<base_date)
)
print("After removing those with two abnormal eGFR")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
)


In [ ]:

# Diag ICD-10 code
########### ############# ############## ############## ###########

print("##### Removing ICD-10 codes")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(pl.col.DIAG_DATE<base_date)
 .drop("DIAG_DATE", "DIAG").unique()
)
display((egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(pl.col.DIAG_DATE<base_date)
 .with_columns(pl.col.DIAG.str.slice(0,3).alias("DIAG"))
)["DIAG"].value_counts().sort("count", descending=True))

print("After removing")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
)


In [ ]:

# Diag ATC code
########### ############# ############## ############## ###########

print("##### Removing ATC codes")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
  # diagnosis
  .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
  .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
  # SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
  .filter(pl.col.MED_DATE<base_date)
  .drop("MED_DATE", "MED").unique()
)

print("After removing")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
    # diagnosis
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
    # SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
)


In [ ]:

# Nephrectomy
########### ############# ############## ############## ###########

print("##### Removing nephrectomy codes")
print_count((egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
    # SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
    .filter(pl.col.APPROX_EVENT_DAY<base_date)
))

print("After removing")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
    # SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
  .filter(~(pl.col.APPROX_EVENT_DAY<base_date)|(pl.col.APPROX_EVENT_DAY.is_null()))
  .drop("APPROX_EVENT_DAY").unique()
)

In [ ]:

# Cystatin C
########### ############# ############## ############## ###########

print("##### Removing Abnormal cystatin C eGFR")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")        
  # Diagnosis codes
  .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
  .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
   # ACE, ARB, or SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
  .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
  .drop("MED_DATE", "MED").unique()
  # Nephro cases
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
  .filter(~(pl.col.APPROX_EVENT_DAY<base_date)|(pl.col.APPROX_EVENT_DAY.is_null()))
  .drop("APPROX_EVENT_DAY").unique()
  # Cystatin C cases
  .filter(pl.col.FINNGENID.is_in(cystc_data.filter(pl.col.DATE<base_date, pl.col.VALUE<60)["FINNGENID"]))
)

print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")    
  # Diagnosis codes
  .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
  .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
   # ACE, ARB, or SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
  .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
  .drop("MED_DATE", "MED").unique()
  # Nephro cases
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
  .filter(~(pl.col.APPROX_EVENT_DAY<base_date)|(pl.col.APPROX_EVENT_DAY.is_null()))
  .drop("APPROX_EVENT_DAY").unique()
  # Cystatin C
  .filter(~pl.col.FINNGENID.is_in(cystc_data.filter(pl.col.DATE<base_date, pl.col.VALUE<60)["FINNGENID"]))
)


In [ ]:

# UACR
########### ############# ############## ############## ###########

print("##### Removing Abnormal urine albumin in urine ratio")
# uacr_diag = (uacr_data
#                 .filter(pl.col.DATE<base_date,pl.col.ABNORM_CUSTOM!=0)
#                 .filter((pl.col.DATE==pl.col.DATE.min()).over("FINNGENID"))
# )
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")    
  # Diagnosis codes
  .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
  .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
   # ACE, ARB, or SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
  .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
  .drop("MED_DATE", "MED").unique()
  # Nephro cases
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
  .filter(~(pl.col.APPROX_EVENT_DAY<base_date)|(pl.col.APPROX_EVENT_DAY.is_null()))
  .drop("APPROX_EVENT_DAY").unique()
  # Cystatin C
  .filter(~pl.col.FINNGENID.is_in(cystc_data.filter(pl.col.DATE<base_date, pl.col.VALUE<60)["FINNGENID"]))
  # UACR
  .filter(pl.col.FINNGENID.is_in(uacr_data.filter(pl.col.DATE<base_date, pl.col.VALUE>=30)["FINNGENID"]))
)

print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")    
  # Diagnosis codes
  .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
  .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
   # ACE, ARB, or SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
  .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
  .drop("MED_DATE", "MED").unique()
  # Nephro cases
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
  .filter(~(pl.col.APPROX_EVENT_DAY<base_date)|(pl.col.APPROX_EVENT_DAY.is_null()))
  .drop("APPROX_EVENT_DAY").unique()
  # Cystatin C
  .filter(~pl.col.FINNGENID.is_in(cystc_data.filter(pl.col.DATE<base_date, pl.col.VALUE<60)["FINNGENID"]))
  # UACR
  .filter(~pl.col.FINNGENID.is_in(uacr_data.filter(pl.col.DATE<base_date, pl.col.VALUE>=30)["FINNGENID"]))
)
# Interestingly the difference here is only 10% left of people with single abnormal UACR -> again back most people with only A10BK


## Excluding

In [ ]:

# Excluding
########### ############# ############## ############## ###########

base_file_name = "egfr_d1_KDIGO-soft2_ld_2025-11-05"
(egfr_data
   # No eGFR based diagnosis
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")   
   # No official ICD-code CKD based diagnosis
  .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
  .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
   # no use of ACE, ARB, or SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
  .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
  .drop("MED_DATE", "MED").unique()
   # No nephrectomy
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
  .filter(~(pl.col.APPROX_EVENT_DAY<base_date)|(pl.col.APPROX_EVENT_DAY.is_null()))
  .drop("APPROX_EVENT_DAY").unique()
  # No abnormal cystatin C eGFR
  .filter(~pl.col.FINNGENID.is_in(cystc_data.filter(pl.col.DATE<base_date, pl.col.VALUE<60)["FINNGENID"]))
  # No abnormal UACR
  .filter(~pl.col.FINNGENID.is_in(uacr_data.filter(pl.col.DATE<base_date, pl.col.VALUE>=30)["FINNGENID"]))
).write_parquet(base_path+"/step2_diags/"+base_file_name+"_filtered_"+get_date()+".parquet")


# Step 3 - Labels

In [ ]:
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                           Step 3 - Labels                       #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
! python3 /home/ivm/valid/scripts/steps/step3_labeling.py \
    --data_path_full "$base_path"/step2_diags/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2".parquet \
    --res_dir "$base_path"/step3_labels/ \
    --lab_name "$lab_name" \
    --start_pred_date "$start_pred_date" --end_pred_date "$end_pred_date" \
    --min_age "$min_age" --max_age "$max_age" \
    --months_buffer "$months_buffer" \
    --abnorm_type "$abnorm_type" \
    --valid_pct "$valid_pct" \
    --test_pct "$test_pct" \
    --version "$version"


# Step 4 - Extra Data

## eGFR Sumstats

In [ ]:

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                           Step 4 - Extra Data                   #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
# eGFR sumstats
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step4_sumstats.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path "$base_path"/step3_labels/ \
    --file_name_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3" \
    --lab_name "$lab_name" \
    --start_date "$base_date" \
    --mean_impute 0

In [ ]:

########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step4_sumstats.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path "$base_path"/step3_labels/ \
    --file_name_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3" \
    --lab_name "$lab_name" \
    --start_date "$base_date" \
    --mean_impute 0 \
    --interpolate 1


## Cyst C Sumstats

In [ ]:
# Cyst C sumstats
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step4_sumstats.py \
  --res_dir "$base_path"/step4_data/ \
  --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
  --file_path_data "$base_path"/step1_clean/"$lab_name_two"_"$extra_two"_"$date_two".parquet \
  --file_path "$base_path"/step3_labels/ \
  --file_name_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"\
  --lab_name "$lab_name" \
  --start_date "$base_date"
